# CADIP staging and AUXIP staging on-demand flows

Demonstration of flows defined on those two stories:   
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-715   
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-718   

In [ ]:
debug_flow = False # For testing only. Should be False in git.

In [ ]:
# For testing, don't commit
import sys
sys.path.insert(0, "/home/ecombelles/workspace/rs-demo/notebooks")
import resources.test_localhost

debug_flow = True

## 1 - Initialisation

In [ ]:
# Access to Prefect
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_staging)

In [ ]:
# Create a test collection
CATALOG_COLLECTION_ID = "SPRINT26_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)

# Check that it is empty
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
assert not list(items)

# Other test values
SESSION_ID = "S1A_20200105072204051312"
CADIP_COLLECTION_ID = "sgs_sentinel1"

In [ ]:
# Other imports
from importlib import reload
import os
import sys
import prefect
from pystac import ItemCollection
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import rs_workflows

# Local paths
rs_workflows_parent = Path(rs_workflows.__path__[0]).parent

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

## 2 - Set up flows parameters

In [ ]:
cadip_flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "cadip_collection_identifier": CADIP_COLLECTION_ID,
  "session_identifier": SESSION_ID,
  "catalog_collection_identifier": CATALOG_COLLECTION_ID
}

auxip_flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "start_datetime": "2024-05-27T09:44:12.509000Z",
  "end_datetime": "2024-05-27T09:44:13.509000Z",
  "eopf_type": "",
  "catalog_collection_identifier": CATALOG_COLLECTION_ID
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

## 3 - Deploy Prefect flows

We deploy the Prefect workflows that are implemented in the rs-client-libraries git repository.

WARNING: the rs-client-libraries source code must be identical in these 3 environments:

- https://github.com/RS-PYTHON/rs-demo.git
- This Jupyter environment
- The Prefect Docker images

In [ ]:
%%bash -s "$rs_workflows_parent"
# Deploy the flows
deploy_file=$(realpath "./cadip_auxip_staging_flows.yaml")
echo "Deploying '$deploy_file'..."
(cd $1; prefect --no-prompt deploy --prefect-file "$deploy_file" --all)

In [ ]:
# Flow deployment names
cadip_deploy = "On-demand Cadip staging/On-demand Cadip staging"
auxip_deploy = "On-demand Auxip staging/On-demand Auxip staging"

# Wait for deployments
for deploy_name in [cadip_deploy, auxip_deploy]:
    await prefect_utils.wait_for_deployment(deploy_name)

In [ ]:
# For testing only: serve from a s3 bucket to test changes more easily
if debug_flow:

    # Use a subfolder named after the current user
    s3_code_folder = f"users/{OWNER_ID}/code" 
    workflows_folder = f"{s3_code_folder}/rs_workflows"
    
    # Upload workflows package and resources contents
    await share_bucket.put_directory(local_path = rs_workflows.__path__[0], to_path = workflows_folder)

    # Reload all rs-client-libraries modules
    for module in list(sys.modules.values()):
        if any(module.__name__.startswith(prefix) for prefix in ["rs_client.", "rs_common.", "rs_workflows."]):
            reload(module)

    # Deploy the flows
    for entrypoint, name, deploy_name in [
        ["on_demand_processing.py:on_demand_auxip_staging", "On-demand Auxip staging", auxip_deploy],
        ["on_demand_processing.py:on_demand_cadip_staging", "On-demand Cadip staging", cadip_deploy],
    ]:
        flow = await prefect.flow.from_source(
            source=share_bucket,
            entrypoint=f"{workflows_folder}/{entrypoint}",
        )
        await flow.deploy(
            name=name,
            work_pool_name=os.environ["PREFECT_WORK_POOL_EOPF"], 
            tags=["debug only"],
            ignore_warnings=True,
        )
        await prefect_utils.wait_for_deployment(deploy_name)

## 4 - Run flows

Run one flow for CADIP staging and one for AUXIP staging.

In [ ]:
# Convert to json to trigger prefect flow
cadip_params_str = to_json(cadip_flow_parameters)
auxip_params_str = to_json(auxip_flow_parameters)

In [ ]:
# Deploy and run flow for CADIP staging
%%bash -s "$cadip_deploy" "$cadip_params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

In [ ]:
# Deploy and run flow for AUXIP staging
%%bash -s "$auxip_deploy" "$auxip_params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

In [ ]:
# Processed items published to the catalog
ItemCollection(list(catalog_client.get_items(CATALOG_COLLECTION_ID)))

## 5 - Shutdown the dask clusters

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## 6 - FOR TESTS ONLY: reset the cluster and run the flows locally from Python

In [ ]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway_staging, None)
    init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *

In [ ]:
if debug_flow:

    # Reload all rs-client-libraries modules
    for module in list(sys.modules.values()):
        if any(module.__name__.startswith(prefix) for prefix in ["rs_client.", "rs_common.", "rs_workflows."]):
            reload(module)

    from rs_workflows import on_demand_processing
    results = await on_demand_processing.on_demand_cadip_staging(**cadip_flow_parameters)
    display(results)
    results = await on_demand_processing.on_demand_auxip_staging(**auxip_flow_parameters)
    display(results)